# Fiche 8 — Requêtes en plusieurs étapes : sous-requêtes et CTE

| | |
|---|---|
| 🎯 **Objectif** | Découper une question en étapes et calculer une part sans piège de division |
| ⏱️ **Durée** | 45 min |
| 🧰 **Prérequis** | Fiches 6 et 7 |
| 🔗 **Dans le projet** | Partie 3 (besoins en proportion, en taux, en comparaison) |

> **L'idée clé.** Une question complexe = plusieurs questions simples. On calcule chaque morceau, on le **nomme**, puis on assemble.

In [1]:
import sqlite3

import pandas as pd

# Ouvre TA base si elle est prête ; sinon, la base de démonstration du dossier (demande-la à ton mentor
# si elle ne t'a pas été transmise). Ouverture en lecture seule : les exemples ne peuvent rien abîmer.
base = sqlite3.connect("file:../data/processed/dataimmo.sqlite?mode=ro", uri=True)


def sql(requete: str) -> pd.DataFrame:
    return pd.read_sql(requete, base)

## 1. ⚠️ Piège : la division entière

En SQL, un entier divisé par un entier donne un **entier** : la partie décimale est jetée.

In [2]:
sql("SELECT 2791 / 34169 AS division_entiere, 2791 * 100.0 / 34169 AS pourcentage")

,division_entiere,pourcentage
0,0,8.168223


`2791 / 34169` vaut 0. Multiplier par `100.0` (un décimal) **avant** la division force un calcul décimal.

## 2. Sous-requête : comparer chaque ligne à un total

Part de chaque type de voie dans l'ensemble des biens (les 5 plus fréquents) :

In [3]:
sql("""
SELECT
    type_voie,
    COUNT(*) AS nb_biens,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM bien), 1) AS part_pct   -- le total, calculé une fois
FROM bien
GROUP BY type_voie
ORDER BY nb_biens DESC
LIMIT 5
""")

,type_voie,nb_biens,part_pct
0,RUE,19580,57.3
1,AV,5107,14.9
2,BD,1842,5.4
3,ALL,1262,3.7
4,NaN,940,2.8


## 3. CTE (`WITH … AS`) : nommer les étapes

Combien de ventes compte, en moyenne, une commune de chaque département ? Étape 1 : compter les ventes par commune.
Étape 2 : faire la moyenne de ces comptes par département.

In [4]:
sql("""
WITH ventes_par_commune AS (                       -- étape 1 : une ligne par commune
    SELECT c.code_departement, c.id_codedep_codecommune, COUNT(*) AS nb_ventes
    FROM vente AS v
    INNER JOIN bien    AS b ON b.id_bien = v.id_bien
    INNER JOIN commune AS c ON c.id_codedep_codecommune = b.id_codedep_codecommune
    GROUP BY c.code_departement, c.id_codedep_codecommune
)
SELECT                                             -- étape 2 : une ligne par département
    code_departement,
    COUNT(*)                  AS nb_communes,
    ROUND(AVG(nb_ventes), 1)  AS ventes_moyennes_par_commune
FROM ventes_par_commune
GROUP BY code_departement
ORDER BY ventes_moyennes_par_commune DESC
LIMIT 5
""")

,code_departement,nb_communes,ventes_moyennes_par_commune
0,75,20,242.2
1,92,35,51.8
2,94,47,41.0
3,93,40,26.9
4,13,48,19.9


Chaque étape se teste seule : remplace le `SELECT` final par `SELECT * FROM ventes_par_commune LIMIT 5` pour voir
le résultat intermédiaire. C'est la façon la plus sûre de déboguer une requête longue.

### ✍️ À toi de jouer

Quelle part des ventes porte sur **plusieurs lots** (`nombre_lots > 1`) ? Donne le nombre de ventes concernées, le total et la part en %, en une seule ligne, avec une CTE et deux sous-requêtes.

In [5]:
# Ta réponse ici : sql(""" ... """)

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```sql
WITH comptes AS (
    SELECT
        (SELECT COUNT(*) FROM vente WHERE nombre_lots > 1) AS plusieurs_lots,
        (SELECT COUNT(*) FROM vente)                       AS total
)
SELECT plusieurs_lots, total, ROUND(plusieurs_lots * 100.0 / total, 1) AS part_pct
FROM comptes;
```

Deux sous-requêtes calculent chacune un nombre ; la CTE les nomme ; la requête finale fait le calcul, avec `100.0` pour éviter la division entière.
</details>

## ✅ À retenir

1. `100.0 *` avant une division d'entiers, sinon le résultat est tronqué.
2. Une sous-requête calcule une valeur de référence (un total) utilisée par la requête principale.
3. Une CTE nomme chaque étape ; on teste chaque étape seule.